In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
import sys
from logging import INFO, StreamHandler, getLogger

logger = getLogger()
if not logger.hasHandlers():
    logger.addHandler(StreamHandler(sys.stdout))
logger.setLevel(INFO)

In [ ]:
import os
import pathlib
from itertools import product
from multiprocessing import Pool
import numpy as np
from tqdm.notebook import tqdm

ROOT_DIR = str(pathlib.Path(".").resolve().parent.parent)
print(f"ROOT_DIR: {ROOT_DIR}")

In [ ]:
sys.path.append(ROOT_DIR)

In [ ]:
from src.synthetic_data.task import (
    calc_results_for_uniform_dist,
    calc_results_for_normal_dist,
    calc_results_for_lognormal_dist,
)
from src.util.io_pickle import write_pickle

In [ ]:
RESULT_DIR = f"{ROOT_DIR}/data/synthetic_data"
os.makedirs(RESULT_DIR, exist_ok=True)

In [ ]:
np.random.seed(42)
np.random.randint(100000, 999999, 5)

In [ ]:
# is_outlier, n_ens, n_times, dist_type, seed
tasks = list(
    product(
        [True, False],
        [10, 20, 40, 60, 80, 100, 150, 200],
        [120, 240, 600, 1_200, 2_400, 6_000, 12_000],
        ["normal", "uniform", "lognormal"],
        [221958, 771155, 231932, 465838, 359178],
    )
)


def process_task(task):
    is_outlier, n_ens, n_times, kind, seed = task

    if kind == "uniform":
        experiment_name = (
            f"Uniform-Ens{n_ens:03}-Out{is_outlier}-Ntime{n_times:05}_Seed{seed:07}"
        )
    elif kind == "normal":
        experiment_name = (
            f"Normal-Ens{n_ens:03}-Out{is_outlier}-Ntime{n_times:05}_Seed{seed:07}"
        )
    elif kind == "lognormal":
        experiment_name = (
            f"Lognormal-Ens{n_ens:03}-Out{is_outlier}-Ntime{n_times:05}_Seed{seed:07}"
        )
    else:
        raise ValueError(f"Unknown kind: {kind}")

    output_path = os.path.join(RESULT_DIR, f"{experiment_name}.pkl")
    if os.path.exists(output_path):
        return

    if kind == "uniform":
        results = calc_results_for_uniform_dist(
            n_ens=n_ens,
            n_times=n_times,
            is_outlier=is_outlier,
            max_nbins=500,
            seed=seed,
        )
    elif kind == "normal":
        results = calc_results_for_normal_dist(
            n_ens=n_ens,
            n_times=n_times,
            is_outlier=is_outlier,
            max_nbins=500,
            seed=seed,
        )
    elif kind == "lognormal":
        results = calc_results_for_lognormal_dist(
            n_ens=n_ens,
            n_times=n_times,
            is_outlier=is_outlier,
            max_nbins=500,
            seed=seed,
        )
    else:
        raise ValueError(f"Unknown kind: {kind}")
    write_pickle(results, output_path)


print(f"total number of tasks = {len(tasks)}")

with Pool(processes=64) as pool, tqdm(total=len(tasks)) as pbar:
    for _ in pool.imap_unordered(process_task, tasks):
        pbar.update()

# Temporal Correlation


In [ ]:
TEMPORAL_CORR_DIR = f"{RESULT_DIR}/temporal_correlation"
os.makedirs(TEMPORAL_CORR_DIR, exist_ok=True)

# phi is the lag-1 autocorrelation per time step (1 month).
# phi = 0 is the existing result without outliers and is not recomputed.
# kind, phi, seed
tasks_temporal_corr = list(
    product(
        ["normal", "uniform", "lognormal"],
        [0.5, 0.9],
        [221958, 771155, 231932, 465838, 359178],
    )
)


def process_task_temporal_corr(task: tuple[str, float, int]) -> None:
    kind, phi, seed = task
    n_ens, n_times = 40, 1_200

    label = {"normal": "Normal", "uniform": "Uniform", "lognormal": "Lognormal"}[kind]
    experiment_name = (
        f"{label}-Ens{n_ens:03}-Phi{phi:.2f}-Ntime{n_times:05}_Seed{seed:07}"
    )
    output_path = os.path.join(TEMPORAL_CORR_DIR, f"{experiment_name}.pkl")
    if os.path.exists(output_path):
        return

    calc_results = {
        "normal": calc_results_for_normal_dist,
        "uniform": calc_results_for_uniform_dist,
        "lognormal": calc_results_for_lognormal_dist,
    }[kind]
    results = calc_results(
        n_ens=n_ens,
        n_times=n_times,
        is_outlier=False,
        max_nbins=500,
        seed=seed,
        phi=phi,
    )
    write_pickle(results, output_path)


n_tasks = len(tasks_temporal_corr)
print(f"total number of tasks = {n_tasks}")

with Pool(processes=min(64, n_tasks)) as pool, tqdm(total=n_tasks) as pbar:
    for _ in pool.imap_unordered(process_task_temporal_corr, tasks_temporal_corr):
        pbar.update()